In [55]:
from kipy import KiCad
from kipy.geometry import Vector2, Angle
from kipy.board_types import (
    BoardLayer,
    BoardSegment,
    BoardRectangle,
    BoardCircle,
    BoardArc,
    BoardPolygon,
    to_concrete_board_shape,
)
from dataclasses import dataclass, field
import numpy as np

if __name__ == "__main__":
    try:
        kicad = KiCad()
        print(f"Connected to KiCad {kicad.get_version()}")
    except BaseException as e:
        print(f"Not connected to KiCad: {e}")

Connected to KiCad 10.0.1 (10.0.1)


In [56]:
@dataclass
class Pin:
    local_pos: np.ndarray
    net: str


@dataclass
class Component:
    id: str
    pos: np.ndarray
    theta: float = 0.0
    half_size: np.ndarray = field(default_factory=lambda: np.array([2.5, 2.5]))
    pins: list = field(default_factory=list)
    fixed: bool = False

In [57]:
board = kicad.get_board()
footprints = board.get_footprints()

for f in footprints:
    print(f"{f.reference_field.text.value}")

R1
R2


In [65]:
# Get board

def n2m(v):
    NM = 1e6  # nanometres per millimetre
    return np.array([v.x, v.y]) / NM

def rect(lo, hi):
    lo, hi = np.asarray(lo, float), np.asarray(hi, float)
    return np.array([lo, [hi[0], lo[1]], hi, [lo[0], hi[1]]])

def get_edge_cuts(board):
    shape = next((s for s in board.get_shapes() if s.layer == BoardLayer.BL_Edge_Cuts), None)
    if not shapes:
        return None
    box = board.get_item_bounding_box(shape)
    if not box: raise Error
    return rect(n2m(box.pos), n2m(box.pos) + n2m(box.size))

def bounding_box

get_edge_cuts(board)

array([[ 38.925, 113.775],
       [116.075, 113.775],
       [116.075, 147.175],
       [ 38.925, 147.175]])

In [68]:
def calc_half_size(board, fp):
    CRTYD = (BoardLayer.BL_F_CrtYd, BoardLayer.BL_B_CrtYd)
    box = None
    for s in fp.definition.shapes:
        if s.layer in CRTYD:
            b = to_concrete_board_shape(s).bounding_box()
            box = b if box is None else (box.merge(b) or box)
        if box is None:
            box = board.get_item_bounding_box(fp)
        return n2
def get_components(board, margin=0.5):
    components = {}
    for fp in board.get_footprints():
        ref = fp.reference_field.text.value
        local = [n2m(p.position) for p in fp.definition.pads]
        pins = [Pin(local_pos=lp, net=p.net.name)
                for lp, p in zip(local, fp.definition.pads)]

        if local:
            local = np.array(local)
            half = np.maximum((local.max(0) - local.min(0)) / 2 + margin, margin)
        else:
            half = np.array([margin, margin])

        components[ref] = Component(
            id=ref,
            pos=n2m(fp.position),
            theta=fp.orientation.to_radians(),
            half_size=half,
            pins=pins,
            fixed=fp.locked,
        )
    return components
get_components(board)

IndentationError: expected an indented block after 'if' statement on line 8 (741359203.py, line 9)

In [64]:
footprints = board.get_footprints()

for f in footprints:
    print(f"ref: {f.reference_field.text.value}")
    print(f"layer: {f.layer}")
    print(f"locked: {f.locked}")
    print(f"orientation: {f.orientation}")
    print()


ref: R1
layer: 3
locked: False
orientation: Angle(0.0)

ref: R2
layer: 3
locked: False
orientation: Angle(0.0)

